# Advanced agents — Multi-agent (subagents)

Lesson `2_3` : a **main agent** that does not do the work itself. It
delegates to two tiny **specialist subagents**, each hidden behind a normal `@tool`.
Same setup as every notebook in this path — only a Gemini key is needed (from `.env`,
Colab Secrets or pasted below): `GOOGLE_API_KEY` — the local project keeps the same key
as `GEMINI_API_KEY`, both are accepted. Same small, cheap model as every notebook in
this path: `gemini-3.1-flash-lite`.

> **Model fix:** the original lesson used an OpenAI model (`gpt-5-nano`) which needs an
> OpenAI key. Like every notebook in this path, this one uses Gemini — no extra key, no extra cost.

**Lesson plan:**
1. **Two specialist tools** — one small tool per job, before any agent exists.
2. **One subagent per tool** — `create_agent` with that single tool.
3. **Wrap the subagents** — ordinary `@tool` wrappers; the main agent never knows they are agents.
4. **Test** — the main agent delegates and reports the answer.
5. **Your turn** — three exercises: the smallest possible version, a printout versus a
   trace, then splitting a real agent of yours.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

def _check_install(pkg: str, mod: str):
    if _ilu.find_spec(mod) is None:
        result = _sp.run([_sys.executable, "-m", "pip", "install", "-q", pkg],
                         capture_output=True, text=True)
        print(f"{pkg}: installed" if result.returncode == 0
              else f"{pkg}: install FAILED -> {result.stderr[-200:]}")
    else:
        print(f"{pkg}: already installed")

for _pkg, _mod in [
    ("python-dotenv", "dotenv"),
    ("langchain", "langchain"),
    ("langgraph", "langgraph"),
    ("langchain-mcp-adapters", "langchain_mcp_adapters"),
    ("google-genai", "google.genai"),
]:
    _check_install(_pkg, _mod)

print("Setup done.")

python-dotenv: already installed
langchain: already installed
langgraph: already installed
langchain-mcp-adapters: already installed
google-genai: already installed
Setup done.


In [2]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Load the local .env file: it lives in the notebooks folder (CWD when the
# notebook is launched by run_notebook.cmd) or one folder up (module root).
# override=True makes the file win over any OS-variable key.
_dotenv = Path.cwd() / ".env"
if not _dotenv.exists():
    _dotenv = Path.cwd().parent / ".env"

if _dotenv.exists():
    load_dotenv(_dotenv, override=True)
    _key_source = f"folder .env ({_dotenv.parent.name}/)"
else:
    load_dotenv(find_dotenv(), override=True)
    _key_source = "OS variable or prompt"

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass(f"Please paste your {var} (hidden): ")

_set_env("GOOGLE_API_KEY")
_set_env("GEMINI_API_KEY")

print("Key source:", _key_source)
print("GOOGLE_API_KEY:", "set" if os.environ.get("GOOGLE_API_KEY") else "MISSING")
print("GEMINI_API_KEY:", "set" if os.environ.get("GEMINI_API_KEY") else "MISSING")
print("Setup done.")

Key source: folder .env (notebooks/)
GOOGLE_API_KEY: set
GEMINI_API_KEY: set
Setup done.


In [3]:
# Same small, cheap Gemini model as every notebook in this path (Module 1 + 2_1).
MODEL = "google_genai:gemini-3.1-flash-lite"
print("Model:", MODEL)

Model: google_genai:gemini-3.1-flash-lite


## Multi-agent — delegating to subagents

Lesson `2_3` splits the work between agents: two tiny *specialist* agents each own one tool,
and a single **main agent** decides which specialist to call. Just like the travel agent let
Kiwi's server do the flight search, this main agent delegates the maths.

> **A subagent is just another agent** — `create_agent` again, with its own tools. The main
> agent reaches it through a normal `@tool` that runs `subagent.invoke(...)`.

### Task 1: two specialist tools


In [4]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

print("Specialist tools:", square_root.name, "+", square.name)


Specialist tools: square_root + square


### Task 2: one subagent per tool

Each subagent is a normal agent with a single specialist tool and the same Gemini model as
everywhere in this notebook.


In [5]:
from langchain.agents import create_agent

# create subagents
subagent_1 = create_agent(
    model=MODEL,
    tools=[square_root],
)

subagent_2 = create_agent(
    model=MODEL,
    tools=[square],
)

print("Subagents ready.")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Subagents ready.


### Task 3: wrap the subagents as tools

The main agent has no idea the helpers are agents: for it, they are two ordinary tools. Each
wrapper forwards the request with `invoke` and returns the subagent's last message.


In [6]:
from langchain.messages import HumanMessage

def _last_text(response):
    msg = response["messages"][-1]
    if isinstance(msg.content, str):
        return msg.content
    return msg.content[0]["text"]

@tool
def call_subagent_1(x: float) -> str:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return _last_text(response)

@tool
def call_subagent_2(x: float) -> str:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return _last_text(response)

main_agent = create_agent(
    model=MODEL,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)

print("Main agent ready.")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Main agent ready.


### Task 4: test — the main agent delegates


In [7]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

_answer = response["messages"][-1]
if isinstance(_answer.content, str):
    print(_answer.content)
else:
    print(_answer.content[0]["text"])


The square root of 456 is approximately 21.3541565040626.


In [8]:
from pprint import pprint

pprint(response)


{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='6b219648-dfe6-46be-bf17-6fd1e1ba3d74'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}, '__gemini_function_call_thought_signatures__': {'call_533003': 'EnMKcQFpFH0T9cyEArGGPfj+xdBX1GElikHCZlHFFWLCYpdsPsmhUK861iJF+nLb7uDDKVFNdraow9ZG2wHFOrdkOZ55pgyd/B61nDbGqdbOxQnmqfTM3EGEfTv9UL/xm7BbYaQqm6MH2lfp8kpinfTsBQP+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0db38-0e95-7502-8254-59359ad7c75a-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_533003', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 19, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='The square 

I never wrote the maths: the main agent picked `call_subagent_1`, which ran `subagent_1`
and returned its answer. Swapping a subagent later (another tool, another prompt, even
another model) would never change the main agent.


### Task 1: the smallest possible version

Build the smallest multi-agent that still has the full shape: **two trivial subagents and one
supervisor**. No real work anywhere — the point is the wiring, not the work. Each subagent is a
boxed prompt (no tools at all), and the supervisor only decides which subagent to call.


In [9]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage

# Two trivial subagents: each is just a boxed prompt with a fixed answer.
north_agent = create_agent(model=MODEL, system_prompt="Answer in exactly two words: North Pole")
south_agent = create_agent(model=MODEL, system_prompt="Answer in exactly two words: South Pole")

def _text(response):
    msg = response["messages"][-1]
    return msg.content if isinstance(msg.content, str) else msg.content[0]["text"]

@tool
def ask_north() -> str:
    """Ask the north specialist."""
    return _text(north_agent.invoke({"messages": [HumanMessage(content="Tell me about the north.")]}))

@tool
def ask_south() -> str:
    """Ask the south specialist."""
    return _text(south_agent.invoke({"messages": [HumanMessage(content="Tell me about the south.")]}))

supervisor = create_agent(
    model=MODEL,
    tools=[ask_north, ask_south],
    system_prompt="You always call the north specialist and report its answer.",
)

question = HumanMessage(content="Which direction should we pick?")
response = supervisor.invoke({"messages": [question]})
print("Answer:", _text(response))


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Answer: The north specialist suggests we head toward the Arctic ice.


### Task 2: print the messages, then read the trace

Two views of the same run. First: the **messages** the run left behind. Then: stream the same
question again and look at the **trace** — the steps of the wiring. Notice how much the
printout did not tell you.


In [10]:
from langchain.messages import AIMessage, ToolMessage

def _show_message(msg, i):
    kind = type(msg).__name__.replace("Message", "")
    if getattr(msg, "tool_calls", None):
        call = msg.tool_calls[0]
        return f"{i}. {kind}: decided -> {call['name']}({call['args']})"
    if isinstance(msg, ToolMessage):
        return f"{i}. {kind} ({msg.name}): {str(msg.content)[:60]}"
    if isinstance(msg, AIMessage) and not isinstance(msg.content, str):
        return f"{i}. {kind}: {msg.content[0]['text'][:60]}"
    return f"{i}. {kind}: {str(getattr(msg, 'content', msg))[:60]}"

print("\n".join(_show_message(m, i) for i, m in enumerate(response["messages"])))


0. Human: Which direction should we pick?
1. AI: decided -> ask_north({'question': 'Which direction should we pick?'})
2. Tool (ask_north): Arctic ice.
3. AI: The north specialist suggests we head toward the Arctic ice.


In [11]:
# The same question again, streamed: each step of the wiring becomes visible.
for update in supervisor.stream({"messages": [question]}, stream_mode="updates"):
    for node, payload in update.items():
        for msg in payload.get("messages", []):
            print(f"[{node}] {_show_message(msg, '*')}")


[model] *. AI: decided -> ask_north({'question': 'Which direction should we pick?'})
[tools] *. Tool (ask_north): Arctic ice.
[model] *. AI: The north specialist suggests "Arctic ice."


**What the printout did not tell you.** The messages show inputs and answers; the trace
shows the *decisions*: the supervisor called `ask_north` on its own, then waited for the
subagent's answer before composing the final sentence. A printout gives the outcome; a trace
gives the wiring.

> 💡 `stream_mode="updates"` is the first level. If a bug hides inside a subagent, trace one
> level deeper — the subagent has its own `model` and `tools` nodes.


### Task 3 : split something real

Take an agent of yours that does too much, and **pull one job out** into a subagent. The steps,
with the lesson above as the reference:

1. Pick one job the agent handles today — a tool, or a whole responsibility.
2. Build the specialist: `create_agent(model=MODEL, tools=[...], system_prompt=...)`.
3. Wrap it: a `@tool` that invokes the subagent and returns its last message (like `ask_north`).
4. Point the main agent at the wrapper instead of the old tool.

Run the same prompts before and after: the answers should not change, but each agent now does
one job. The wrapper is the contract — later you can swap the specialist (another prompt,
another model) without touching the orchestrator.


#### Minimal way to finish Task 3

`get_weather` stands in for **your old tool** — faked below so this cell really runs. The four
steps of the recipe above, in the smallest possible form:


In [12]:
# The old tool that used to live inside one big agent (faked, so this cell really runs):
@tool
def get_weather(city: str) -> str:
    """Fake weather lookup for a city."""
    return f"{city}: sunny, 26 C"

# 1. Make the specialist
weather_agent = create_agent(
    model=MODEL,
    tools=[get_weather],            # your old tool
    system_prompt="You are a weather specialist.",
)

# 2. Wrap it as a tool
@tool
def ask_weather(city: str) -> str:
    """Ask the weather specialist about a city."""
    response = weather_agent.invoke(
        {"messages": [HumanMessage(content=f"Weather in {city}?")]}
    )
    return _text(response)          # last message, text only

# 3. Give the wrapper to the main agent
main_agent = create_agent(
    model=MODEL,
    tools=[ask_weather],            # instead of get_weather
    system_prompt="Delegate weather questions to the specialist.",
)

# 4. Test
response = main_agent.invoke({"messages": [HumanMessage(content="Weather in Algiers?")]})
print(_text(response))


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


The current weather in Algiers is sunny with a temperature of 26°C.


## What I learned

1. **A subagent is just another agent.** Same `create_agent`, its own one tool — the only special thing is that it hides behind a tool.
2. **The wrapper is the wiring.** A normal `@tool` calls `subagent.invoke(...)` and returns the last message; the main agent never knows it is talking to an agent.
3. **Delegation makes specialists swappable.** Replace a subagent (another tool, another prompt, even another model) without touching the main agent.
4. **A printout is not a trace.** Printing the messages shows the outcome; `agent.stream(..., stream_mode="updates")` shows every decision — which tool, with which argument, from which node.
5. **Start from the smallest version.** Two toy subagents and one supervisor teach the wiring before any real work gets in the way.


## Summary of functions

| Function / class | Simple definition |
|---|---|
| `create_agent(MODEL, tools=[specialist])` | A tiny specialist subagent, usually hidden behind a tool. |
| `@tool` + `subagent.invoke({...})` | The bridge: a normal tool that runs a subagent and returns its last message. |
| `agent.stream(..., stream_mode="updates")` | Streams each step of a run — the trace of the wiring, with the node name. |


By
> Kassouar Fatima                   26-09-2026
